# Mission 10 · 4차 오분류 처방 실험 · Kaggle GPU판

3차 실험의 우승 조건을 고정합니다.

```text
clean preprocessing
MAX_LEN = 300
```

4차에서는 다음 네 결과를 같은 5-fold split에서 비교합니다.

| 이름 | 구성 | 역할 |
|---|---|---|
| `baseline` | Packed Attention BiGRU | 5-fold 기준선 |
| `coarse` | BiGRU + coarse auxiliary head | 큰 분야를 가로지르는 오류 억제 |
| `tfidf` | word/character n-gram linear branch | 정확한 topic phrase 보강 |
| `ensemble` | coarse 0.7 + TF-IDF 0.3 | 의미 문맥과 phrase 표현 결합 |

선택 기준은 **5-fold OOF Macro F1**입니다.  
공식 test set은 이 notebook의 모델 선택 과정에서 사용하지 않습니다.


## 0. Kaggle 실행 전 설정

### GPU

Kaggle 오른쪽 설정:

```text
Accelerator: GPU
Internet: On
```

`fetch_20newsgroups()` 최초 다운로드에는 Internet이 필요합니다.

### Google Drive 자동 checkpoint 저장

GPU session이 끊겨도 이어서 실행할 수 있도록 checkpoint와 결과를 Google Drive에 동기화합니다.

인증 우선순위:

1. Kaggle Secret `GDRIVE_OAUTH_TOKEN_JSON`
2. Kaggle Secret `GDRIVE_SERVICE_ACCOUNT_JSON`
3. Secret이 없거나 인증이 실패하면 `/kaggle/working`에만 저장하고 학습은 계속 진행

#### OAuth token 방식

로컬에서 생성한 Google OAuth `token.json` 전체를 Kaggle Secret
`GDRIVE_OAUTH_TOKEN_JSON`에 넣습니다.

자동 갱신을 위해 다음 항목이 포함돼야 합니다.

```text
refresh_token
client_id
client_secret
token_uri
```

#### Service Account 방식

1. Google Cloud에서 Service Account key JSON을 생성합니다.
2. JSON의 `client_email`을 아래 Drive folder에 Editor로 공유합니다.
3. JSON 전체를 Kaggle Secret `GDRIVE_SERVICE_ACCOUNT_JSON`에 넣습니다.

```text
Drive parent folder ID:
1o8DfSBPM7HV3knsWVCQQ1WmDSJWlLdec
```

개인 My Drive에서 Service Account storage quota가 막히면 OAuth token 방식을 사용합니다.

자동 동기화:

```text
매 epoch
- latest checkpoint
- history
- 새 best checkpoint

fold 완료
- best/latest checkpoint
- validation probability
- validation index
- fold 결과 JSON

전체 완료
- OOF probability
- CSV와 class report
- confusion matrix
- summary
```

시작 시 Drive의 `mission10_stage4_outputs`를 `/kaggle/working`으로 복구한 뒤 resume합니다.


In [ ]:
# Kaggle image에 package가 없다면 아래 줄의 주석을 해제한 뒤 한 번 실행합니다.
# !pip install -q gensim google-api-python-client google-auth


In [ ]:
import gc
import json
import mimetypes
import os
import random
import re
import shutil
import sys
import time
import warnings

from collections import Counter
from contextlib import nullcontext
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from gensim.models import FastText
from IPython.display import display, Markdown
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
SEED = 42

# Data
TEST_SIZE = 0.15
N_SPLITS = 5
MAX_LEN = 300
MIN_FREQ = 2
MAX_VOCAB_SIZE = 40_000

# FastText
EMBED_DIM = 100
FASTTEXT_EPOCHS = 10
FASTTEXT_WORKERS = min(
    4,
    max((os.cpu_count() or 2) - 1, 1),
)

# Neural model
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.4
LABEL_SMOOTHING = 0.05
COARSE_LABEL_SMOOTHING = 0.02
COARSE_LOSS_WEIGHT = 0.3

# Optimization
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 7
MIN_DELTA = 0.001

SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-5
GRAD_CLIP_NORM = 1.0

# Stage 4 branches
RUN_NEURAL_VARIANTS = [
    "baseline",
    "coarse",
]
RUN_TFIDF_BRANCH = True
NEURAL_ENSEMBLE_WEIGHT = 0.7
TFIDF_ENSEMBLE_WEIGHT = 0.3

# Resume behavior
SKIP_COMPLETED_FOLDS = True
RESUME_INCOMPLETE_FOLD = True

# Google Drive checkpoint sync
ENABLE_GOOGLE_DRIVE_SYNC = True
GDRIVE_PARENT_FOLDER_ID = "1o8DfSBPM7HV3knsWVCQQ1WmDSJWlLdec"
GDRIVE_RUN_FOLDER_NAME = "mission10_stage4_outputs"
GDRIVE_OAUTH_SECRET_NAME = "GDRIVE_OAUTH_TOKEN_JSON"
GDRIVE_SERVICE_ACCOUNT_SECRET_NAME = "GDRIVE_SERVICE_ACCOUNT_JSON"
GDRIVE_SYNC_EVERY_N_EPOCHS = 1
GDRIVE_RESTORE_ON_START = True

# Device-safe batch settings
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    BATCH_SIZE = 32
    GRAD_ACCUMULATION_STEPS = 4
    NUM_WORKERS = min(2, os.cpu_count() or 1)
    PIN_MEMORY = True
    GPU_NAME = torch.cuda.get_device_name(0)
    USE_AMP = "P100" not in GPU_NAME
else:
    DEVICE = torch.device("cpu")
    BATCH_SIZE = 16
    GRAD_ACCUMULATION_STEPS = 8
    NUM_WORKERS = 0
    PIN_MEMORY = False
    USE_AMP = False

OUTPUT_DIR = (
    Path("/kaggle/working/mission10_stage4_outputs")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "mission10_stage4_outputs"
)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULT_DIR = OUTPUT_DIR / "results"
CACHE_DIR = OUTPUT_DIR / "cache"
PROBABILITY_DIR = OUTPUT_DIR / "probabilities"
PLOT_DIR = OUTPUT_DIR / "plots"

for directory in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    RESULT_DIR,
    CACHE_DIR,
    PROBABILITY_DIR,
    PLOT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("DEVICE:", DEVICE)
print("USE_AMP:", USE_AMP)
print("Batch size:", BATCH_SIZE)
print(
    "Effective batch size:",
    BATCH_SIZE * GRAD_ACCUMULATION_STEPS,
)
print("Output:", OUTPUT_DIR)


In [ ]:
# 이전 Kaggle output dataset을 input으로 연결했을 때 자동 복구
def restore_previous_output():
    if not Path("/kaggle/input").exists():
        return None

    candidates = list(
        Path("/kaggle/input").glob(
            "**/mission10_stage4_outputs"
        )
    )

    if not candidates:
        print(
            "연결된 이전 Stage 4 output이 없습니다."
        )
        return None

    source = candidates[0]

    print(
        "이전 output 복구:",
        source,
    )

    shutil.copytree(
        source,
        OUTPUT_DIR,
        dirs_exist_ok=True,
    )

    return source


restore_previous_output()


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def release_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def format_elapsed(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(
        seconds,
        3600,
    )
    minutes, seconds = divmod(
        remainder,
        60,
    )

    if hours:
        return (
            f"{hours:d}h "
            f"{minutes:02d}m "
            f"{seconds:02d}s"
        )

    return (
        f"{minutes:02d}m "
        f"{seconds:02d}s"
    )


def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )
    except (TypeError, AttributeError):
        return torch.cuda.amp.GradScaler(
            enabled=USE_AMP,
        )


def torch_load_compat(
    path,
    map_location=DEVICE,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


seed_everything(SEED)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision(
        "high"
    )


## 0-1. Google Drive checkpoint sync와 resume

Drive 오류가 발생해도 local checkpoint는 유지되고 training은 계속됩니다.


In [ ]:
GDRIVE_SCOPES = [
    "https://www.googleapis.com/auth/drive",
]

DRIVE_SERVICE = None
DRIVE_SYNC_READY = False
GDRIVE_REMOTE_ROOT_ID = None

_DRIVE_FOLDER_CACHE = {}
_DRIVE_FILE_CACHE = {}


def read_kaggle_secret(name):
    value = os.environ.get(name)

    if value:
        return value

    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def escape_drive_query(value):
    return str(value).replace(
        "'",
        "\\'",
    )


def find_drive_child(
    parent_id,
    name,
    mime_type=None,
):
    cache_key = (
        str(parent_id),
        str(name),
        str(mime_type),
    )

    if cache_key in _DRIVE_FILE_CACHE:
        return _DRIVE_FILE_CACHE[cache_key]

    parts = [
        f"'{escape_drive_query(parent_id)}' in parents",
        f"name = '{escape_drive_query(name)}'",
        "trashed = false",
    ]

    if mime_type:
        parts.append(
            f"mimeType = '{escape_drive_query(mime_type)}'"
        )

    response = (
        DRIVE_SERVICE.files()
        .list(
            q=" and ".join(parts),
            spaces="drive",
            fields="files(id,name,mimeType,size)",
            pageSize=10,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        )
        .execute()
    )

    files = response.get("files", [])
    found = files[0] if files else None

    if found:
        _DRIVE_FILE_CACHE[cache_key] = found

    return found


def get_or_create_drive_folder(
    parent_id,
    name,
):
    cache_key = (
        str(parent_id),
        str(name),
    )

    if cache_key in _DRIVE_FOLDER_CACHE:
        return _DRIVE_FOLDER_CACHE[cache_key]

    folder_mime = (
        "application/vnd.google-apps.folder"
    )

    found = find_drive_child(
        parent_id,
        name,
        mime_type=folder_mime,
    )

    if found:
        folder_id = found["id"]
    else:
        created = (
            DRIVE_SERVICE.files()
            .create(
                body={
                    "name": name,
                    "mimeType": folder_mime,
                    "parents": [parent_id],
                },
                fields="id",
                supportsAllDrives=True,
            )
            .execute()
        )
        folder_id = created["id"]

    _DRIVE_FOLDER_CACHE[cache_key] = folder_id

    return folder_id


def drive_parent_for_relative_path(
    relative_parent,
):
    current_id = GDRIVE_REMOTE_ROOT_ID

    for part in Path(relative_parent).parts:
        if part in ("", "."):
            continue

        current_id = get_or_create_drive_folder(
            current_id,
            part,
        )

    return current_id


def init_google_drive_sync():
    global DRIVE_SERVICE
    global DRIVE_SYNC_READY
    global GDRIVE_REMOTE_ROOT_ID

    if not ENABLE_GOOGLE_DRIVE_SYNC:
        print("Google Drive sync: disabled")
        return False

    try:
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google.oauth2 import service_account
        from googleapiclient.discovery import build

        oauth_json = read_kaggle_secret(
            GDRIVE_OAUTH_SECRET_NAME
        )

        if oauth_json:
            credentials = (
                Credentials.from_authorized_user_info(
                    json.loads(oauth_json),
                    scopes=GDRIVE_SCOPES,
                )
            )
            auth_type = "OAuth token"

            if (
                credentials.expired
                and credentials.refresh_token
            ):
                credentials.refresh(Request())
        else:
            service_json = read_kaggle_secret(
                GDRIVE_SERVICE_ACCOUNT_SECRET_NAME
            )

            if not service_json:
                print(
                    "Drive sync inactive: Kaggle Secret "
                    f"{GDRIVE_OAUTH_SECRET_NAME} 또는 "
                    f"{GDRIVE_SERVICE_ACCOUNT_SECRET_NAME} 필요"
                )
                return False

            credentials = (
                service_account.Credentials
                .from_service_account_info(
                    json.loads(service_json),
                    scopes=GDRIVE_SCOPES,
                )
            )
            auth_type = "Service Account"

        DRIVE_SERVICE = build(
            "drive",
            "v3",
            credentials=credentials,
            cache_discovery=False,
        )

        GDRIVE_REMOTE_ROOT_ID = (
            get_or_create_drive_folder(
                GDRIVE_PARENT_FOLDER_ID,
                GDRIVE_RUN_FOLDER_NAME,
            )
        )

        DRIVE_SYNC_READY = True

        print(
            "Google Drive sync ready |",
            auth_type,
            "| folder id:",
            GDRIVE_REMOTE_ROOT_ID,
        )

        return True

    except Exception as error:
        DRIVE_SERVICE = None
        DRIVE_SYNC_READY = False
        GDRIVE_REMOTE_ROOT_ID = None

        print(
            "Google Drive sync 초기화 실패:",
            repr(error),
        )
        print(
            "training은 /kaggle/working 저장으로 계속됩니다."
        )

        return False


def sync_file_to_google_drive(
    local_path,
):
    if not DRIVE_SYNC_READY:
        return False

    from googleapiclient.http import MediaFileUpload

    local_path = Path(local_path)

    if not local_path.is_file():
        return False

    try:
        relative_path = local_path.relative_to(
            OUTPUT_DIR
        )
    except ValueError:
        relative_path = Path(local_path.name)

    parent_id = drive_parent_for_relative_path(
        relative_path.parent
    )

    remote_name = relative_path.name

    mime_type = (
        mimetypes.guess_type(
            str(local_path)
        )[0]
        or "application/octet-stream"
    )

    existing = find_drive_child(
        parent_id,
        remote_name,
    )

    media = MediaFileUpload(
        str(local_path),
        mimetype=mime_type,
        resumable=True,
    )

    if existing:
        response = (
            DRIVE_SERVICE.files()
            .update(
                fileId=existing["id"],
                media_body=media,
                fields="id,name,size",
                supportsAllDrives=True,
            )
            .execute()
        )
    else:
        response = (
            DRIVE_SERVICE.files()
            .create(
                body={
                    "name": remote_name,
                    "parents": [parent_id],
                },
                media_body=media,
                fields="id,name,size",
                supportsAllDrives=True,
            )
            .execute()
        )

        _DRIVE_FILE_CACHE[
            (
                str(parent_id),
                remote_name,
                str(None),
            )
        ] = response

    return True


def safe_sync_files(
    paths,
    label=None,
):
    if not DRIVE_SYNC_READY:
        return False

    try:
        synced = 0

        for path in paths:
            if (
                path is not None
                and sync_file_to_google_drive(path)
            ):
                synced += 1

        if label:
            print(
                f"Drive sync [{label}]: "
                f"{synced} file(s)"
            )

        return True

    except Exception as error:
        print(
            "Google Drive sync 경고:",
            repr(error),
        )
        print(
            "local checkpoint를 유지하며 "
            "training을 계속합니다."
        )

        return False


def safe_sync_output_tree(
    label="full output",
):
    files = sorted(
        path
        for path in OUTPUT_DIR.rglob("*")
        if path.is_file()
    )

    return safe_sync_files(
        files,
        label=label,
    )


def list_drive_children(
    parent_id,
):
    page_token = None
    children = []

    while True:
        response = (
            DRIVE_SERVICE.files()
            .list(
                q=(
                    f"'{escape_drive_query(parent_id)}' "
                    "in parents and trashed = false"
                ),
                spaces="drive",
                fields=(
                    "nextPageToken, "
                    "files(id,name,mimeType,size)"
                ),
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            )
            .execute()
        )

        children.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return children


def download_drive_file(
    file_id,
    local_path,
):
    from googleapiclient.http import MediaIoBaseDownload

    local_path = Path(local_path)
    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        DRIVE_SERVICE.files()
        .get_media(fileId=file_id)
    )

    with local_path.open("wb") as output:
        downloader = MediaIoBaseDownload(
            output,
            request,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()


def restore_drive_folder(
    remote_folder_id,
    local_folder,
):
    folder_mime = (
        "application/vnd.google-apps.folder"
    )

    restored = 0

    for item in list_drive_children(
        remote_folder_id
    ):
        local_path = (
            Path(local_folder)
            / item["name"]
        )

        if item["mimeType"] == folder_mime:
            local_path.mkdir(
                parents=True,
                exist_ok=True,
            )

            restored += restore_drive_folder(
                item["id"],
                local_path,
            )
        else:
            download_drive_file(
                item["id"],
                local_path,
            )

            restored += 1

    return restored


def restore_output_from_google_drive():
    if (
        not DRIVE_SYNC_READY
        or not GDRIVE_RESTORE_ON_START
    ):
        return 0

    try:
        restored = restore_drive_folder(
            GDRIVE_REMOTE_ROOT_ID,
            OUTPUT_DIR,
        )

        print(
            "Google Drive output 복구:",
            f"{restored} file(s)",
        )

        return restored

    except Exception as error:
        print(
            "Google Drive 복구 경고:",
            repr(error),
        )
        print(
            "현재 local output으로 계속합니다."
        )

        return 0


init_google_drive_sync()
restore_output_from_google_drive()


## 1. Data load와 고정 test 분리

전체 18,846개 중 15% test를 먼저 고정하고, 나머지 85%만 5-fold CV pool로 사용합니다.

```text
전체
├─ test 15%: 마지막 최종 평가용
└─ CV pool 85%: 4차 모델 선택용
```

이 notebook에서는 test prediction을 만들지 않습니다.


In [ ]:
news_data = fetch_20newsgroups(
    subset="all",
    remove=(
        "headers",
        "footers",
        "quotes",
    ),
    shuffle=True,
    random_state=SEED,
)

texts = np.asarray(
    news_data.data,
    dtype=object,
)
labels = np.asarray(
    news_data.target,
    dtype=np.int64,
)
target_names = list(
    news_data.target_names
)

(
    cv_texts,
    test_texts,
    cv_labels,
    test_labels,
) = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=SEED,
)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

fold_splits = list(
    skf.split(
        cv_texts,
        cv_labels,
    )
)

print("전체:", len(texts))
print("CV pool:", len(cv_texts))
print("고정 test:", len(test_texts))
print("Classes:", len(target_names))

for fold, (train_idx, val_idx) in enumerate(
    fold_splits,
    start=1,
):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}"
    )


## 2. 3차 우승 전처리: clean + MAX_LEN 300


In [ ]:
TOKEN_PATTERN = re.compile(
    r"[a-z]+(?:'[a-z]+)?"
)

UUENCODE_BEGIN_PATTERN = re.compile(
    r"^\s*begin\s+[0-7]{3}\s+\S+",
    flags=re.IGNORECASE,
)
UUENCODE_END_PATTERN = re.compile(
    r"^\s*end\s*$",
    flags=re.IGNORECASE,
)
BASE64_LINE_PATTERN = re.compile(
    r"^[A-Za-z0-9+/=]+$"
)


def tokenize_original(text):
    text = text.lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
    )
    text = re.sub(
        r"\b[\w.\-+]+@[\w.\-]+\.\w+\b",
        " ",
        text,
    )
    text = re.sub(
        r"[^a-z'\s]",
        " ",
        text,
    )
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return TOKEN_PATTERN.findall(text)


def looks_like_encoded_line(line):
    stripped = line.strip()

    if len(stripped) < 80:
        return False

    compact = re.sub(
        r"\s+",
        "",
        stripped,
    )

    if len(compact) < 80:
        return False

    base64_like = (
        len(compact) >= 100
        and BASE64_LINE_PATTERN.fullmatch(
            compact
        )
        is not None
    )

    symbol_count = sum(
        not character.isalnum()
        for character in compact
    )
    symbol_ratio = (
        symbol_count
        / max(len(compact), 1)
    )

    uuencode_like = (
        stripped.startswith("M")
        and len(compact) >= 60
        and symbol_ratio >= 0.20
    )

    symbol_heavy = (
        len(compact) >= 120
        and symbol_ratio >= 0.35
    )

    return (
        base64_like
        or uuencode_like
        or symbol_heavy
    )


def remove_encoded_payload(text):
    retained_lines = []
    inside_uuencode_block = False

    for line in text.splitlines():
        if UUENCODE_BEGIN_PATTERN.match(
            line
        ):
            inside_uuencode_block = True
            continue

        if inside_uuencode_block:
            if UUENCODE_END_PATTERN.match(
                line
            ):
                inside_uuencode_block = False
            continue

        if looks_like_encoded_line(line):
            continue

        retained_lines.append(line)

    return "\n".join(retained_lines)


def tokenize_clean(text):
    text = remove_encoded_payload(text)
    tokens = tokenize_original(text)

    token_counts = Counter(tokens)

    repeated_short_noise = {
        token
        for token, count
        in token_counts.items()
        if (
            len(token) <= 2
            and count >= 20
            and (
                count
                / max(len(tokens), 1)
            ) >= 0.20
        )
    }

    if token_counts.get("ax", 0) >= 8:
        repeated_short_noise.add("ax")

    if repeated_short_noise:
        tokens = [
            token
            for token in tokens
            if token not in repeated_short_noise
        ]

    return tokens


tokenize_started_at = perf_counter()

cv_tokens = [
    tokenize_clean(text)
    for text in tqdm(
        cv_texts,
        desc="Clean tokenization",
    )
]

# TF-IDF branch가 같은 clean corpus를 사용하도록 문자열로 다시 묶음
cv_clean_texts = np.asarray(
    [
        " ".join(tokens)
        for tokens in cv_tokens
    ],
    dtype=object,
)

token_lengths = np.asarray(
    [
        len(tokens)
        for tokens in cv_tokens
    ]
)

print(
    "Tokenization time:",
    format_elapsed(
        perf_counter()
        - tokenize_started_at
    ),
)
print(
    "Mean tokens:",
    f"{token_lengths.mean():.2f}",
)
print(
    "Truncated over 300:",
    f"{np.mean(token_lengths > MAX_LEN):.2%}",
)


## 3. Coarse label 구성


In [ ]:
COARSE_GROUPS = {
    "computer": [
        "comp.graphics",
        "comp.os.ms-windows.misc",
        "comp.sys.ibm.pc.hardware",
        "comp.sys.mac.hardware",
        "comp.windows.x",
    ],
    "recreation": [
        "rec.autos",
        "rec.motorcycles",
        "rec.sport.baseball",
        "rec.sport.hockey",
    ],
    "science": [
        "sci.crypt",
        "sci.electronics",
        "sci.med",
        "sci.space",
    ],
    "politics": [
        "talk.politics.guns",
        "talk.politics.mideast",
        "talk.politics.misc",
    ],
    "religion": [
        "alt.atheism",
        "soc.religion.christian",
        "talk.religion.misc",
    ],
    "misc": [
        "misc.forsale",
    ],
}

COARSE_NAMES = list(
    COARSE_GROUPS
)

fine_name_to_coarse_id = {}

for coarse_id, coarse_name in enumerate(
    COARSE_NAMES
):
    for fine_name in COARSE_GROUPS[
        coarse_name
    ]:
        fine_name_to_coarse_id[
            fine_name
        ] = coarse_id

fine_id_to_coarse_id = np.asarray(
    [
        fine_name_to_coarse_id[
            fine_name
        ]
        for fine_name in target_names
    ],
    dtype=np.int64,
)

cv_coarse_labels = (
    fine_id_to_coarse_id[
        cv_labels
    ]
)

coarse_mapping_df = pd.DataFrame(
    {
        "fine_class": target_names,
        "coarse_class": [
            COARSE_NAMES[
                fine_id_to_coarse_id[
                    fine_id
                ]
            ]
            for fine_id in range(
                len(target_names)
            )
        ],
    }
)

display(coarse_mapping_df)


## 4. Fold별 vocabulary와 FastText

엄밀한 CV를 위해 vocabulary와 FastText는 각 fold의 training 문서만 사용합니다.  
Validation 문서는 vocabulary와 FastText 학습에 들어가지 않습니다.


In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

FOLD_RESOURCE_CACHE = {}


def build_vocabulary(
    train_token_documents,
):
    counter = Counter(
        token
        for document
        in train_token_documents
        for token in document
    )

    common_tokens = [
        token
        for token, frequency
        in counter.most_common(
            MAX_VOCAB_SIZE - 2
        )
        if frequency >= MIN_FREQ
    ]

    itos = [
        PAD_TOKEN,
        UNK_TOKEN,
    ] + common_tokens

    stoi = {
        token: index
        for index, token in enumerate(
            itos
        )
    }

    return stoi, itos, counter


def train_or_load_fold_fasttext(
    fold,
    train_token_documents,
):
    model_path = (
        CACHE_DIR
        / f"fasttext_fold_{fold}.model"
    )

    if model_path.exists():
        print(
            "FastText cache load:",
            model_path.name,
        )
        return FastText.load(
            str(model_path)
        )

    print(
        "FastText train:",
        model_path.name,
    )

    model = FastText(
        sentences=train_token_documents,
        vector_size=EMBED_DIM,
        window=5,
        min_count=MIN_FREQ,
        workers=FASTTEXT_WORKERS,
        sg=1,
        negative=10,
        epochs=FASTTEXT_EPOCHS,
        seed=SEED + fold,
    )

    model.save(
        str(model_path)
    )

    return model


def build_embedding_matrix(
    fasttext_model,
    stoi,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    matrix = rng.normal(
        loc=0.0,
        scale=0.05,
        size=(
            len(stoi),
            EMBED_DIM,
        ),
    ).astype(np.float32)

    matrix[PAD_IDX] = 0.0

    for token, index in stoi.items():
        if index in (
            PAD_IDX,
            UNK_IDX,
        ):
            continue

        matrix[index] = (
            fasttext_model.wv.get_vector(
                token
            )
        )

    return matrix


def encode_documents(
    token_documents,
    stoi,
):
    encoded = []

    for tokens in token_documents:
        token_ids = [
            stoi.get(
                token,
                UNK_IDX,
            )
            for token in tokens[:MAX_LEN]
        ]

        if not token_ids:
            token_ids = [UNK_IDX]

        encoded.append(token_ids)

    return encoded


def prepare_fold_resources(
    fold,
    train_idx,
    val_idx,
):
    if fold in FOLD_RESOURCE_CACHE:
        return FOLD_RESOURCE_CACHE[
            fold
        ]

    started_at = perf_counter()

    train_tokens = [
        cv_tokens[index]
        for index in train_idx
    ]
    val_tokens = [
        cv_tokens[index]
        for index in val_idx
    ]

    stoi, itos, counter = (
        build_vocabulary(
            train_tokens
        )
    )

    fasttext_model = (
        train_or_load_fold_fasttext(
            fold,
            train_tokens,
        )
    )

    embedding_matrix = (
        build_embedding_matrix(
            fasttext_model,
            stoi,
            seed=SEED + fold,
        )
    )

    train_sequences = (
        encode_documents(
            train_tokens,
            stoi,
        )
    )
    val_sequences = (
        encode_documents(
            val_tokens,
            stoi,
        )
    )

    resource = {
        "stoi": stoi,
        "itos": itos,
        "counter": counter,
        "embedding_matrix": (
            embedding_matrix
        ),
        "train_sequences": (
            train_sequences
        ),
        "val_sequences": (
            val_sequences
        ),
        "train_fine_labels": (
            cv_labels[train_idx]
        ),
        "val_fine_labels": (
            cv_labels[val_idx]
        ),
        "train_coarse_labels": (
            cv_coarse_labels[train_idx]
        ),
        "val_coarse_labels": (
            cv_coarse_labels[val_idx]
        ),
        "val_idx": np.asarray(
            val_idx
        ),
    }

    FOLD_RESOURCE_CACHE[
        fold
    ] = resource

    print(
        f"Fold {fold} resources | "
        f"vocab={len(stoi):,} | "
        f"time={format_elapsed(perf_counter() - started_at)}"
    )

    return resource


## 5. Dataset과 DataLoader

각 sequence는 Dataset 생성 시 tensor로 한 번만 변환합니다.  
Kaggle CUDA에서는 `pin_memory=True`, `num_workers=2`를 사용하고 CPU fallback에서는 안전 설정으로 바뀝니다.


In [ ]:
class NewsDataset(Dataset):
    def __init__(
        self,
        sequences,
        fine_labels,
        coarse_labels,
    ):
        self.sequences = [
            torch.as_tensor(
                sequence,
                dtype=torch.long,
            )
            for sequence in sequences
        ]

        self.fine_labels = (
            torch.as_tensor(
                fine_labels,
                dtype=torch.long,
            )
        )
        self.coarse_labels = (
            torch.as_tensor(
                coarse_labels,
                dtype=torch.long,
            )
        )

    def __len__(self):
        return len(
            self.fine_labels
        )

    def __getitem__(self, index):
        return (
            self.sequences[index],
            self.fine_labels[index],
            self.coarse_labels[index],
        )


def collate_batch(batch):
    (
        sequences,
        fine_labels,
        coarse_labels,
    ) = zip(*batch)

    lengths = torch.tensor(
        [
            len(sequence)
            for sequence in sequences
        ],
        dtype=torch.long,
    )

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    return (
        padded_sequences,
        lengths,
        torch.stack(fine_labels),
        torch.stack(coarse_labels),
    )


def make_fold_loaders(resource):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    common = {
        "batch_size": BATCH_SIZE,
        "collate_fn": collate_batch,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "persistent_workers": (
            NUM_WORKERS > 0
        ),
    }

    train_loader = DataLoader(
        NewsDataset(
            resource[
                "train_sequences"
            ],
            resource[
                "train_fine_labels"
            ],
            resource[
                "train_coarse_labels"
            ],
        ),
        shuffle=True,
        generator=generator,
        **common,
    )

    val_loader = DataLoader(
        NewsDataset(
            resource[
                "val_sequences"
            ],
            resource[
                "val_fine_labels"
            ],
            resource[
                "val_coarse_labels"
            ],
        ),
        shuffle=False,
        **common,
    )

    return (
        train_loader,
        val_loader,
    )


## 6. Baseline / Coarse auxiliary model


In [ ]:
class MultiTaskPackedAttentionBiGRU(
    nn.Module
):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim,
        num_fine_classes,
        num_coarse_classes,
        use_coarse_head,
    ):
        super().__init__()

        embedding_tensor = torch.tensor(
            embedding_matrix,
            dtype=torch.float32,
        )

        self.use_coarse_head = (
            use_coarse_head
        )

        self.embedding = (
            nn.Embedding.from_pretrained(
                embedding_tensor,
                freeze=False,
                padding_idx=PAD_IDX,
            )
        )

        self.gru = nn.GRU(
            input_size=(
                embedding_tensor.shape[1]
            ),
            hidden_size=hidden_dim,
            num_layers=NUM_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=(
                DROPOUT
                if NUM_LAYERS > 1
                else 0.0
            ),
        )

        self.attention_score = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                hidden_dim,
            ),
            nn.Tanh(),
            nn.Linear(
                hidden_dim,
                1,
                bias=False,
            ),
        )

        self.norm = nn.LayerNorm(
            hidden_dim * 2
        )
        self.dropout = nn.Dropout(
            DROPOUT
        )

        self.fine_classifier = nn.Linear(
            hidden_dim * 2,
            num_fine_classes,
        )

        if self.use_coarse_head:
            self.coarse_classifier = nn.Linear(
                hidden_dim * 2,
                num_coarse_classes,
            )
        else:
            self.coarse_classifier = None

    def encode_document(
        self,
        input_ids,
        lengths,
    ):
        embedded = self.embedding(
            input_ids
        )

        packed_input = (
            pack_padded_sequence(
                embedded,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
        )

        packed_output, _ = self.gru(
            packed_input
        )

        sequence_output, _ = (
            pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=input_ids.size(1),
            )
        )

        attention_logits = (
            self.attention_score(
                sequence_output
            ).squeeze(-1)
        )

        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device,
        ).unsqueeze(0)

        valid_mask = (
            positions
            < lengths.to(
                input_ids.device
            ).unsqueeze(1)
        )

        attention_logits = (
            attention_logits.masked_fill(
                ~valid_mask,
                torch.finfo(attention_logits.dtype).min,
            )
        )

        attention_weights = torch.softmax(
            attention_logits,
            dim=1,
        )

        document_vector = torch.sum(
            sequence_output
            * attention_weights.unsqueeze(-1),
            dim=1,
        )

        document_vector = self.norm(
            document_vector
        )
        document_vector = self.dropout(
            document_vector
        )

        return document_vector

    def forward(
        self,
        input_ids,
        lengths,
    ):
        document_vector = (
            self.encode_document(
                input_ids,
                lengths,
            )
        )

        fine_logits = (
            self.fine_classifier(
                document_vector
            )
        )

        coarse_logits = None

        if self.coarse_classifier is not None:
            coarse_logits = (
                self.coarse_classifier(
                    document_vector
                )
            )

        return (
            fine_logits,
            coarse_logits,
        )


## 7. Metrics, evaluation, checkpoint resume


In [ ]:
def compute_metrics(
    y_true,
    y_pred,
):
    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    (
        precision,
        recall,
        f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(
            precision
        ),
        "macro_recall": float(
            recall
        ),
        "macro_f1": float(f1),
    }


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    fine_criterion,
):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    for (
        input_ids,
        lengths,
        fine_labels,
        _,
    ) in data_loader:
        input_ids = input_ids.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        fine_labels = fine_labels.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        with amp_context():
            (
                fine_logits,
                _,
            ) = model(
                input_ids,
                lengths,
            )

            loss = fine_criterion(
                fine_logits,
                fine_labels,
            )

        probabilities = torch.softmax(
            fine_logits.float(),
            dim=1,
        )

        total_loss += (
            loss.item()
            * fine_labels.size(0)
        )

        all_labels.append(
            fine_labels.cpu().numpy()
        )
        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    labels_array = np.concatenate(
        all_labels
    )
    probabilities_array = np.concatenate(
        all_probabilities
    )
    predictions_array = (
        probabilities_array.argmax(
            axis=1
        )
    )

    metrics = compute_metrics(
        labels_array,
        predictions_array,
    )

    average_loss = (
        total_loss
        / len(data_loader.dataset)
    )

    return (
        float(average_loss),
        metrics,
        labels_array,
        predictions_array,
        probabilities_array,
    )


def save_json(data, path):
    with Path(path).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


def train_neural_fold(
    variant,
    fold,
    train_idx,
    val_idx,
):
    use_coarse_head = (
        variant == "coarse"
    )

    fold_name = (
        f"{variant}_fold_{fold}"
    )

    record_path = (
        RESULT_DIR
        / f"{fold_name}.json"
    )
    probability_path = (
        PROBABILITY_DIR
        / f"{fold_name}_val_probs.npy"
    )
    index_path = (
        PROBABILITY_DIR
        / f"{fold_name}_val_idx.npy"
    )
    best_path = (
        CHECKPOINT_DIR
        / f"{fold_name}_best.pt"
    )
    latest_path = (
        CHECKPOINT_DIR
        / f"{fold_name}_latest.pt"
    )
    history_path = (
        RESULT_DIR
        / f"{fold_name}_history.json"
    )

    if (
        SKIP_COMPLETED_FOLDS
        and record_path.exists()
        and probability_path.exists()
        and index_path.exists()
    ):
        print(
            "완료된 fold load:",
            fold_name,
        )

        with record_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            record = json.load(file)

        return (
            record,
            np.load(index_path),
            np.load(probability_path),
        )

    seed_everything(
        SEED + fold
    )
    release_memory()

    resource = prepare_fold_resources(
        fold,
        train_idx,
        val_idx,
    )

    (
        train_loader,
        val_loader,
    ) = make_fold_loaders(
        resource
    )

    model = MultiTaskPackedAttentionBiGRU(
        embedding_matrix=resource[
            "embedding_matrix"
        ],
        hidden_dim=HIDDEN_DIM,
        num_fine_classes=len(
            target_names
        ),
        num_coarse_classes=len(
            COARSE_NAMES
        ),
        use_coarse_head=(
            use_coarse_head
        ),
    ).to(DEVICE)

    fine_criterion = (
        nn.CrossEntropyLoss(
            label_smoothing=(
                LABEL_SMOOTHING
            )
        )
    )
    coarse_criterion = (
        nn.CrossEntropyLoss(
            label_smoothing=(
                COARSE_LABEL_SMOOTHING
            )
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
            threshold=MIN_DELTA,
            threshold_mode="abs",
            min_lr=MIN_LR,
        )
    )

    scaler = make_grad_scaler()

    best_val_f1 = -np.inf
    best_epoch = 0
    no_improvement_epochs = 0
    start_epoch = 1

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_f1": [],
        "learning_rate": [],
        "epoch_seconds": [],
    }

    if (
        RESUME_INCOMPLETE_FOLD
        and latest_path.exists()
        and not record_path.exists()
    ):
        print(
            "중간 checkpoint resume:",
            latest_path.name,
        )

        state = torch_load_compat(
            latest_path
        )

        model.load_state_dict(
            state["model_state"]
        )
        optimizer.load_state_dict(
            state["optimizer_state"]
        )
        scheduler.load_state_dict(
            state["scheduler_state"]
        )

        if (
            USE_AMP
            and state.get(
                "scaler_state"
            )
        ):
            scaler.load_state_dict(
                state["scaler_state"]
            )

        best_val_f1 = float(
            state["best_val_f1"]
        )
        best_epoch = int(
            state["best_epoch"]
        )
        no_improvement_epochs = int(
            state[
                "no_improvement_epochs"
            ]
        )
        history = state["history"]
        start_epoch = int(
            state["epoch"]
        ) + 1

    fold_started_at = perf_counter()

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1,
    ):
        epoch_started_at = perf_counter()

        model.train()
        optimizer.zero_grad(
            set_to_none=True
        )

        total_train_loss = 0.0
        total_batches = len(
            train_loader
        )
        remainder = (
            total_batches
            % GRAD_ACCUMULATION_STEPS
        )

        progress = tqdm(
            enumerate(train_loader),
            total=total_batches,
            desc=(
                f"{fold_name} "
                f"epoch {epoch}/{MAX_EPOCHS}"
            ),
            leave=False,
        )

        for batch_index, (
            input_ids,
            lengths,
            fine_labels,
            coarse_labels,
        ) in progress:
            input_ids = input_ids.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )
            fine_labels = fine_labels.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )
            coarse_labels = coarse_labels.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            with amp_context():
                (
                    fine_logits,
                    coarse_logits,
                ) = model(
                    input_ids,
                    lengths,
                )

                fine_loss = fine_criterion(
                    fine_logits,
                    fine_labels,
                )

                raw_loss = fine_loss

                if (
                    use_coarse_head
                    and coarse_logits is not None
                ):
                    coarse_loss = (
                        coarse_criterion(
                            coarse_logits,
                            coarse_labels,
                        )
                    )

                    raw_loss = (
                        fine_loss
                        + COARSE_LOSS_WEIGHT
                        * coarse_loss
                    )

            if (
                remainder > 0
                and batch_index
                >= total_batches - remainder
            ):
                accumulation_divisor = (
                    remainder
                )
            else:
                accumulation_divisor = (
                    GRAD_ACCUMULATION_STEPS
                )

            scaled_loss = (
                raw_loss
                / accumulation_divisor
            )

            scaler.scale(
                scaled_loss
            ).backward()

            should_step = (
                (
                    batch_index + 1
                )
                % GRAD_ACCUMULATION_STEPS
                == 0
                or (
                    batch_index + 1
                )
                == total_batches
            )

            if should_step:
                scaler.unscale_(
                    optimizer
                )

                grad_norm = (
                    nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=(
                            GRAD_CLIP_NORM
                        ),
                    )
                )

                scaler.step(
                    optimizer
                )
                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

            total_train_loss += (
                raw_loss.item()
                * fine_labels.size(0)
            )

            progress.set_postfix(
                loss=(
                    f"{raw_loss.item():.4f}"
                ),
                lr=(
                    f"{optimizer.param_groups[0]['lr']:.1e}"
                ),
            )

        train_loss = (
            total_train_loss
            / len(train_loader.dataset)
        )

        (
            val_loss,
            val_metrics,
            _,
            _,
            _,
        ) = evaluate_model(
            model,
            val_loader,
            fine_criterion,
        )

        scheduler.step(
            val_metrics["macro_f1"]
        )

        current_lr = (
            optimizer.param_groups[0][
                "lr"
            ]
        )
        epoch_seconds = (
            perf_counter()
            - epoch_started_at
        )

        history["train_loss"].append(
            float(train_loss)
        )
        history["val_loss"].append(
            float(val_loss)
        )
        history["val_accuracy"].append(
            val_metrics["accuracy"]
        )
        history["val_macro_f1"].append(
            val_metrics["macro_f1"]
        )
        history["learning_rate"].append(
            float(current_lr)
        )
        history["epoch_seconds"].append(
            float(epoch_seconds)
        )

        improved = (
            val_metrics["macro_f1"]
            > best_val_f1 + MIN_DELTA
        )

        if improved:
            best_val_f1 = (
                val_metrics["macro_f1"]
            )
            best_epoch = epoch
            no_improvement_epochs = 0

            torch.save(
                model.state_dict(),
                best_path,
            )
        else:
            no_improvement_epochs += 1

        latest_state = {
            "epoch": epoch,
            "model_state": (
                model.state_dict()
            ),
            "optimizer_state": (
                optimizer.state_dict()
            ),
            "scheduler_state": (
                scheduler.state_dict()
            ),
            "scaler_state": (
                scaler.state_dict()
                if USE_AMP
                else None
            ),
            "best_val_f1": (
                best_val_f1
            ),
            "best_epoch": (
                best_epoch
            ),
            "no_improvement_epochs": (
                no_improvement_epochs
            ),
            "history": history,
        }

        torch.save(
            latest_state,
            latest_path,
        )
        save_json(
            history,
            history_path,
        )

        if (
            epoch
            % GDRIVE_SYNC_EVERY_N_EPOCHS
            == 0
        ):
            epoch_sync_paths = [
                latest_path,
                history_path,
            ]

            if improved:
                epoch_sync_paths.append(
                    best_path
                )

            safe_sync_files(
                epoch_sync_paths,
                label=(
                    f"{fold_name} "
                    f"epoch {epoch}"
                ),
            )

        print(
            f"[{fold_name}] "
            f"epoch={epoch:03d} | "
            f"train={train_loss:.4f} | "
            f"val={val_loss:.4f} | "
            f"val F1={val_metrics['macro_f1']:.4f} | "
            f"best={best_val_f1:.4f} | "
            f"lr={current_lr:.1e} | "
            f"time={format_elapsed(epoch_seconds)}"
        )

        if (
            no_improvement_epochs
            >= EARLY_STOP_PATIENCE
        ):
            print(
                f"{fold_name} early stop "
                f"at epoch {epoch}"
            )
            break

    if not best_path.exists():
        raise RuntimeError(
            f"Best checkpoint가 없습니다: "
            f"{best_path}"
        )

    model.load_state_dict(
        torch_load_compat(
            best_path
        )
    )

    (
        best_val_loss,
        best_val_metrics,
        val_true,
        val_pred,
        val_probabilities,
    ) = evaluate_model(
        model,
        val_loader,
        fine_criterion,
    )

    np.save(
        probability_path,
        val_probabilities,
    )
    np.save(
        index_path,
        np.asarray(val_idx),
    )

    record = {
        "variant": variant,
        "fold": int(fold),
        "best_epoch": int(
            best_epoch
        ),
        "val_loss": float(
            best_val_loss
        ),
        **best_val_metrics,
        "device": str(DEVICE),
        "amp": bool(USE_AMP),
        "fold_seconds": float(
            perf_counter()
            - fold_started_at
        ),
        "checkpoint": str(
            best_path
        ),
    }

    save_json(
        record,
        record_path,
    )

    safe_sync_files(
        [
            best_path,
            latest_path,
            history_path,
            record_path,
            probability_path,
            index_path,
        ],
        label=f"{fold_name} complete",
    )

    # best와 latest를 모두 남겨
    # resume와 audit에 사용합니다.
    model.to("cpu")
    del model
    release_memory()

    return (
        record,
        np.asarray(val_idx),
        val_probabilities,
    )


## 8. Neural 5-fold 실행

GPU가 끊긴 뒤 다시 실행해도 완료된 fold는 probability와 JSON을 불러옵니다.


In [ ]:
def run_neural_cv(variant):
    oof_probabilities = np.full(
        (
            len(cv_texts),
            len(target_names),
        ),
        np.nan,
        dtype=np.float32,
    )

    records = []
    cv_started_at = perf_counter()

    for fold, (
        train_idx,
        val_idx,
    ) in enumerate(
        fold_splits,
        start=1,
    ):
        print("\n" + "=" * 100)
        print(
            f"Neural variant={variant} "
            f"| fold={fold}/{N_SPLITS}"
        )
        print("=" * 100)

        (
            record,
            loaded_val_idx,
            val_probabilities,
        ) = train_neural_fold(
            variant,
            fold,
            train_idx,
            val_idx,
        )

        oof_probabilities[
            loaded_val_idx
        ] = val_probabilities

        records.append(record)

        partial_oof_path = (
            PROBABILITY_DIR
            / f"{variant}_oof_partial.npy"
        )

        np.save(
            partial_oof_path,
            oof_probabilities,
        )

        safe_sync_files(
            [partial_oof_path],
            label=(
                f"{variant} partial OOF "
                f"after fold {fold}"
            ),
        )

    if np.isnan(
        oof_probabilities
    ).any():
        raise RuntimeError(
            f"{variant} OOF probability에 "
            "빈 값이 있습니다."
        )

    predictions = (
        oof_probabilities.argmax(
            axis=1
        )
    )

    metrics = compute_metrics(
        cv_labels,
        predictions,
    )

    records_df = pd.DataFrame(
        records
    )

    records_df.to_csv(
        RESULT_DIR
        / f"{variant}_fold_results.csv",
        index=False,
    )

    np.save(
        PROBABILITY_DIR
        / f"{variant}_oof.npy",
        oof_probabilities,
    )

    summary = {
        "variant": variant,
        **metrics,
        "fold_macro_f1_mean": float(
            records_df[
                "macro_f1"
            ].mean()
        ),
        "fold_macro_f1_std": float(
            records_df[
                "macro_f1"
            ].std(ddof=1)
        ),
        "total_seconds": float(
            perf_counter()
            - cv_started_at
        ),
    }

    save_json(
        summary,
        RESULT_DIR
        / f"{variant}_oof_summary.json",
    )

    display(
        records_df.style.format(
            {
                "val_loss": "{:.4f}",
                "accuracy": "{:.4f}",
                "macro_precision": "{:.4f}",
                "macro_recall": "{:.4f}",
                "macro_f1": "{:.4f}",
                "fold_seconds": "{:.1f}",
            }
        )
    )

    safe_sync_output_tree(
        label=f"{variant} CV complete"
    )

    print(
        f"{variant} OOF Macro F1:",
        f"{metrics['macro_f1']:.4f}",
    )
    print(
        "Fold Macro F1:",
        f"{summary['fold_macro_f1_mean']:.4f} "
        f"± {summary['fold_macro_f1_std']:.4f}",
    )

    return (
        oof_probabilities,
        records_df,
        summary,
    )


neural_outputs = {}

for variant in RUN_NEURAL_VARIANTS:
    neural_outputs[
        variant
    ] = run_neural_cv(
        variant
    )


## 9. TF-IDF word / character n-gram branch

이 branch는 GPU를 사용하지 않습니다.  
`SGDClassifier(loss="log_loss")`로 class probability를 만들고 neural probability와 결합합니다.


In [ ]:
def make_tfidf_pipeline(
    fold,
):
    features = FeatureUnion(
        [
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.98,
                    sublinear_tf=True,
                    max_features=80_000,
                    dtype=np.float32,
                ),
            ),
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    sublinear_tf=True,
                    max_features=120_000,
                    dtype=np.float32,
                ),
            ),
        ],
        n_jobs=min(
            2,
            os.cpu_count() or 1,
        ),
    )

    classifier = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=50,
        tol=1e-3,
        average=True,
        random_state=(
            SEED + fold
        ),
        n_jobs=-1,
    )

    return (
        features,
        classifier,
    )


def run_tfidf_cv():
    oof_probabilities = np.full(
        (
            len(cv_texts),
            len(target_names),
        ),
        np.nan,
        dtype=np.float32,
    )

    records = []

    for fold, (
        train_idx,
        val_idx,
    ) in enumerate(
        fold_splits,
        start=1,
    ):
        fold_name = (
            f"tfidf_fold_{fold}"
        )

        record_path = (
            RESULT_DIR
            / f"{fold_name}.json"
        )
        probability_path = (
            PROBABILITY_DIR
            / f"{fold_name}_val_probs.npy"
        )
        index_path = (
            PROBABILITY_DIR
            / f"{fold_name}_val_idx.npy"
        )

        if (
            SKIP_COMPLETED_FOLDS
            and record_path.exists()
            and probability_path.exists()
            and index_path.exists()
        ):
            print(
                "완료된 TF-IDF fold load:",
                fold_name,
            )

            with record_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                record = json.load(file)

            loaded_val_idx = np.load(
                index_path
            )
            val_probabilities = np.load(
                probability_path
            )

        else:
            started_at = perf_counter()

            (
                features,
                classifier,
            ) = make_tfidf_pipeline(
                fold
            )

            train_features = (
                features.fit_transform(
                    cv_clean_texts[
                        train_idx
                    ]
                )
            )
            val_features = (
                features.transform(
                    cv_clean_texts[
                        val_idx
                    ]
                )
            )

            classifier.fit(
                train_features,
                cv_labels[train_idx],
            )

            val_probabilities = (
                classifier.predict_proba(
                    val_features
                ).astype(np.float32)
            )

            # class 순서를 0~19로 보장
            if not np.array_equal(
                classifier.classes_,
                np.arange(
                    len(target_names)
                ),
            ):
                reordered = np.zeros(
                    (
                        len(val_idx),
                        len(target_names),
                    ),
                    dtype=np.float32,
                )
                reordered[
                    :,
                    classifier.classes_,
                ] = val_probabilities
                val_probabilities = (
                    reordered
                )

            val_predictions = (
                val_probabilities.argmax(
                    axis=1
                )
            )
            metrics = compute_metrics(
                cv_labels[val_idx],
                val_predictions,
            )

            record = {
                "variant": "tfidf",
                "fold": int(fold),
                **metrics,
                "fold_seconds": float(
                    perf_counter()
                    - started_at
                ),
            }

            save_json(
                record,
                record_path,
            )
            np.save(
                probability_path,
                val_probabilities,
            )
            np.save(
                index_path,
                np.asarray(val_idx),
            )

            safe_sync_files(
                [
                    record_path,
                    probability_path,
                    index_path,
                ],
                label=f"{fold_name} complete",
            )

            loaded_val_idx = (
                np.asarray(val_idx)
            )

            del (
                features,
                classifier,
                train_features,
                val_features,
            )
            release_memory()

        oof_probabilities[
            loaded_val_idx
        ] = val_probabilities

        records.append(record)

        print(
            f"[{fold_name}] "
            f"F1={record['macro_f1']:.4f} | "
            f"time={format_elapsed(record['fold_seconds'])}"
        )

    if np.isnan(
        oof_probabilities
    ).any():
        raise RuntimeError(
            "TF-IDF OOF probability에 "
            "빈 값이 있습니다."
        )

    predictions = (
        oof_probabilities.argmax(
            axis=1
        )
    )
    metrics = compute_metrics(
        cv_labels,
        predictions,
    )

    records_df = pd.DataFrame(
        records
    )

    records_df.to_csv(
        RESULT_DIR
        / "tfidf_fold_results.csv",
        index=False,
    )
    np.save(
        PROBABILITY_DIR
        / "tfidf_oof.npy",
        oof_probabilities,
    )

    summary = {
        "variant": "tfidf",
        **metrics,
        "fold_macro_f1_mean": float(
            records_df[
                "macro_f1"
            ].mean()
        ),
        "fold_macro_f1_std": float(
            records_df[
                "macro_f1"
            ].std(ddof=1)
        ),
    }

    save_json(
        summary,
        RESULT_DIR
        / "tfidf_oof_summary.json",
    )

    safe_sync_output_tree(
        label="TF-IDF CV complete"
    )

    print(
        "TF-IDF OOF Macro F1:",
        f"{metrics['macro_f1']:.4f}",
    )

    return (
        oof_probabilities,
        records_df,
        summary,
    )


tfidf_output = None

if RUN_TFIDF_BRANCH:
    tfidf_output = run_tfidf_cv()


## 10. OOF 결과와 ensemble 비교

Ensemble weight는 우선 고정값을 사용합니다.

```text
coarse neural = 0.7
TF-IDF        = 0.3
```

test 결과를 보면서 weight를 바꾸지 않습니다.


In [ ]:
def summarize_probability_model(
    name,
    probabilities,
):
    predictions = probabilities.argmax(
        axis=1
    )

    metrics = compute_metrics(
        cv_labels,
        predictions,
    )

    return {
        "model": name,
        **metrics,
    }


summary_rows = []
probability_models = {}

for variant, output in neural_outputs.items():
    probabilities = output[0]

    probability_models[
        variant
    ] = probabilities

    summary_rows.append(
        summarize_probability_model(
            variant,
            probabilities,
        )
    )

if tfidf_output is not None:
    tfidf_probabilities = (
        tfidf_output[0]
    )

    probability_models[
        "tfidf"
    ] = tfidf_probabilities

    summary_rows.append(
        summarize_probability_model(
            "tfidf",
            tfidf_probabilities,
        )
    )

if (
    "coarse" in probability_models
    and "tfidf" in probability_models
):
    ensemble_probabilities = (
        NEURAL_ENSEMBLE_WEIGHT
        * probability_models[
            "coarse"
        ]
        + TFIDF_ENSEMBLE_WEIGHT
        * probability_models[
            "tfidf"
        ]
    )

    ensemble_probabilities = (
        ensemble_probabilities
        / ensemble_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    probability_models[
        "ensemble"
    ] = ensemble_probabilities

    np.save(
        PROBABILITY_DIR
        / "ensemble_oof.npy",
        ensemble_probabilities,
    )

    summary_rows.append(
        summarize_probability_model(
            "ensemble",
            ensemble_probabilities,
        )
    )

stage4_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    stage4_summary_df.style.format(
        {
            "accuracy": "{:.4f}",
            "macro_precision": "{:.4f}",
            "macro_recall": "{:.4f}",
            "macro_f1": "{:.4f}",
        }
    )
)

stage4_summary_df.to_csv(
    RESULT_DIR
    / "stage4_oof_model_comparison.csv",
    index=False,
)

best_model_name = (
    stage4_summary_df.iloc[0][
        "model"
    ]
)

print(
    "OOF 기준 우승:",
    best_model_name,
)


safe_sync_output_tree(
    label="OOF model comparison"
)


## 11. 오분류 진단과 class별 report


In [ ]:
def make_top_confusions(
    y_true,
    y_pred,
):
    count_cm = confusion_matrix(
        y_true,
        y_pred,
    )
    normalized_cm = confusion_matrix(
        y_true,
        y_pred,
        normalize="true",
    )

    rows = []

    for true_index, true_name in enumerate(
        target_names
    ):
        for predicted_index, predicted_name in enumerate(
            target_names
        ):
            if (
                true_index
                == predicted_index
            ):
                continue

            count = int(
                count_cm[
                    true_index,
                    predicted_index,
                ]
            )

            if count == 0:
                continue

            rows.append(
                {
                    "true_class": true_name,
                    "predicted_class": (
                        predicted_name
                    ),
                    "count": count,
                    "rate": float(
                        normalized_cm[
                            true_index,
                            predicted_index,
                        ]
                    ),
                }
            )

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "count",
                "rate",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


diagnostic_tables = {}

for model_name, probabilities in (
    probability_models.items()
):
    predictions = probabilities.argmax(
        axis=1
    )

    report_df = (
        pd.DataFrame(
            classification_report(
                cv_labels,
                predictions,
                target_names=target_names,
                output_dict=True,
                zero_division=0,
            )
        )
        .T
        .loc[
            target_names,
            [
                "precision",
                "recall",
                "f1-score",
                "support",
            ],
        ]
        .sort_values(
            "f1-score"
        )
    )

    top_confusions_df = (
        make_top_confusions(
            cv_labels,
            predictions,
        )
    )

    diagnostic_tables[
        model_name
    ] = {
        "report": report_df,
        "confusions": (
            top_confusions_df
        ),
    }

    report_df.to_csv(
        RESULT_DIR
        / f"{model_name}_class_report.csv"
    )
    top_confusions_df.to_csv(
        RESULT_DIR
        / f"{model_name}_top_confusions.csv",
        index=False,
    )

    print(
        "\n",
        "=" * 90,
    )
    print(
        f"{model_name}: "
        "낮은 F1 class"
    )
    display(
        report_df.head(8).style.format(
            {
                "precision": "{:.4f}",
                "recall": "{:.4f}",
                "f1-score": "{:.4f}",
                "support": "{:.0f}",
            }
        )
    )

    print(
        f"{model_name}: "
        "상위 confusion"
    )
    display(
        top_confusions_df.head(
            15
        ).style.format(
            {
                "rate": "{:.2%}",
            }
        )
    )


best_probabilities = (
    probability_models[
        best_model_name
    ]
)
best_predictions = (
    best_probabilities.argmax(
        axis=1
    )
)

normalized_cm = confusion_matrix(
    cv_labels,
    best_predictions,
    normalize="true",
)

fig, ax = plt.subplots(
    figsize=(15, 15)
)

ConfusionMatrixDisplay(
    confusion_matrix=normalized_cm,
    display_labels=target_names,
).plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=90,
    values_format=".2f",
    colorbar=False,
)

plt.title(
    f"Stage 4 OOF Normalized "
    f"Confusion Matrix: "
    f"{best_model_name}"
)
plt.tight_layout()

plot_path = (
    PLOT_DIR
    / f"{best_model_name}_oof_confusion.png"
)

plt.savefig(
    plot_path,
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print(
    "Confusion matrix saved:",
    plot_path,
)


safe_sync_output_tree(
    label="diagnostics and plots"
)


## 12. OOF 오분류 원문 보기

OOF prediction이므로 각 문서는 자신을 학습에 사용하지 않은 fold model의 예측을 받았습니다.


In [ ]:
def show_oof_confusion_examples(
    model_name,
    true_class,
    predicted_class,
    n=5,
    max_chars=1200,
):
    probabilities = (
        probability_models[
            model_name
        ]
    )
    predictions = probabilities.argmax(
        axis=1
    )

    true_id = target_names.index(
        true_class
    )
    predicted_id = target_names.index(
        predicted_class
    )

    indices = np.flatnonzero(
        (cv_labels == true_id)
        & (predictions == predicted_id)
    )

    print(
        f"{model_name} | "
        f"{true_class} → "
        f"{predicted_class} | "
        f"{len(indices)}건"
    )

    for number, index in enumerate(
        indices[:n],
        start=1,
    ):
        confidence = float(
            probabilities[
                index,
                predicted_id,
            ]
        )

        print("\n" + "=" * 100)
        print(
            f"예시 {number} | "
            f"OOF index={index} | "
            f"confidence={confidence:.4f}"
        )
        print(
            str(cv_texts[index])[
                :max_chars
            ]
        )


# 예시
# show_oof_confusion_examples(
#     best_model_name,
#     "talk.religion.misc",
#     "alt.atheism",
# )
# show_oof_confusion_examples(
#     best_model_name,
#     "alt.atheism",
#     "sci.med",
# )
# show_oof_confusion_examples(
#     best_model_name,
#     "talk.politics.misc",
#     "sci.med",
# )


## 13. 자동 결론 저장


## 13-1. 수동 Google Drive 전체 동기화

자동 sync가 일시적으로 실패했거나 중간 백업이 필요할 때 실행합니다.


In [ ]:
safe_sync_output_tree(label="manual full sync")


In [ ]:
best_row = (
    stage4_summary_df.iloc[0]
)

conclusion_lines = [
    "# Mission 10 Stage 4 결론",
    "",
    "## 고정 데이터 조건",
    "- clean preprocessing",
    "- MAX_LEN=300",
    "",
    "## 5-fold OOF 결과",
]

for _, row in (
    stage4_summary_df.iterrows()
):
    conclusion_lines.append(
        f"- {row['model']}: "
        f"Accuracy={row['accuracy']:.4f}, "
        f"Macro F1={row['macro_f1']:.4f}"
    )

conclusion_lines.extend(
    [
        "",
        "## 선택",
        (
            f"- OOF 기준 우승 모델: "
            f"**{best_model_name}**"
        ),
        (
            f"- OOF Macro F1: "
            f"**{best_row['macro_f1']:.4f}**"
        ),
        "",
        "## 해석 원칙",
        (
            "- coarse head가 개선되면 "
            "religion/politics/science처럼 "
            "큰 분야를 가로지르는 오류를 "
            "보조 loss가 줄인 것으로 해석합니다."
        ),
        (
            "- TF-IDF 또는 ensemble이 개선되면 "
            "word/character n-gram의 정확한 "
            "topic phrase가 하위 class 구분을 "
            "보강한 것으로 해석합니다."
        ),
        (
            "- 공식 test set은 Stage 4 설정을 "
            "확정한 뒤 마지막 한 번만 평가합니다."
        ),
        "",
        "## GPU session 주의",
        (
            "- 결과 폴더를 Kaggle output으로 "
            "저장한 뒤 다음 session에서 input "
            "dataset으로 연결하면 fold resume가 "
            "가능합니다."
        ),
    ]
)

conclusion_text = "\n".join(
    conclusion_lines
)

display(
    Markdown(conclusion_text)
)

conclusion_path = (
    RESULT_DIR
    / "stage4_summary.md"
)

conclusion_path.write_text(
    conclusion_text,
    encoding="utf-8",
)

safe_sync_output_tree(
    label="Stage 4 final output"
)

print(
    "모든 결과:",
    OUTPUT_DIR,
)
print(
    "Google Drive sync ready:",
    DRIVE_SYNC_READY,
)
print(
    "Google Drive folder id:",
    GDRIVE_REMOTE_ROOT_ID,
)


## 14. Stage 4 실제 결과와 결론

5-fold OOF 결과는 다음과 같습니다.

| 모델 | Accuracy | Macro F1 | Baseline 대비 |
|---|---:|---:|---:|
| **TF-IDF** | **0.7523** | **0.7424** | **+0.0213** |
| Ensemble | 0.7454 | 0.7373 | +0.0161 |
| Coarse auxiliary BiGRU | 0.7306 | 0.7224 | +0.0013 |
| Baseline BiGRU | 0.7316 | 0.7211 | 기준 |

### 해석

- `coarse auxiliary head`의 개선폭은 `+0.0013`으로 매우 작았습니다.
- `word/character TF-IDF`는 `+0.0213`으로 가장 분명한 개선을 보였습니다.
- 현재 ensemble은 `coarse 0.7 + TF-IDF 0.3`이므로, 성능이 낮은 neural branch의 비중이 지나치게 커서 TF-IDF 단독보다 낮았습니다.
- 따라서 Stage 4의 우승 설정은 **TF-IDF**입니다.
- 공식 test set은 아직 사용하지 않습니다. Stage 5의 설정까지 OOF로 확정한 뒤 마지막에 한 번만 평가합니다.

### 실행 안정성 메모

- Tesla P100에서는 AMP를 자동으로 끕니다.
- attention mask 값은 dtype에 맞춰 `torch.finfo(dtype).min`을 사용하므로 FP16 overflow를 방지합니다.


## 15. 우승 모델 TF-IDF의 취약 class 진단

낮은 F1 class는 다음과 같습니다.

| class | Precision | Recall | F1 |
|---|---:|---:|---:|
| `talk.religion.misc` | 0.5315 | 0.3315 | **0.4083** |
| `talk.politics.misc` | 0.6481 | 0.6009 | **0.6236** |
| `alt.atheism` | 0.6687 | 0.6333 | **0.6505** |
| `rec.autos` | 0.5799 | 0.7634 | **0.6591** |
| `sci.electronics` | 0.6848 | 0.7069 | **0.6957** |
| `comp.sys.ibm.pc.hardware` | 0.7138 | 0.6838 | **0.6985** |

가장 큰 confusion은 다음과 같습니다.

- `talk.religion.misc → soc.religion.christian`: 123건, 23.0%
- `talk.religion.misc → alt.atheism`: 72건, 13.5%
- `talk.politics.misc → talk.politics.guns`: 90건, 13.7%
- `comp.sys.ibm.pc.hardware → comp.os.ms-windows.misc`: 73건, 8.7%
- `rec.motorcycles → rec.autos`: 68건, 8.0%

`misc` class는 주제가 넓고 인접 class와 vocabulary를 공유하므로 단일 global classifier만으로 경계를 만들기 어렵습니다.  
따라서 다음 단계에서는 전체 모델을 더 복잡하게 만드는 것보다 **혼동군별 specialist classifier**가 더 직접적인 처방입니다.


In [ ]:
# 실제 실행 결과에서 취약 class와 우선 confusion을 다시 생성합니다.
winner_name = (
    best_model_name
    if "best_model_name" in globals()
    else "tfidf"
)

winner_report = (
    diagnostic_tables[winner_name]["report"]
    .copy()
    .sort_values("f1-score")
)

winner_confusions = (
    diagnostic_tables[winner_name]["confusions"]
    .copy()
)

hard_class_table = (
    winner_report
    .head(8)
    .reset_index()
    .rename(columns={"index": "class"})
)

priority_confusions = (
    winner_confusions
    .head(20)
    .reset_index(drop=True)
)

display(
    hard_class_table.style.format(
        {
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1-score": "{:.4f}",
            "support": "{:.0f}",
        }
    )
)

display(
    priority_confusions.style.format(
        {"rate": "{:.2%}"}
    )
)

hard_class_table.to_csv(
    RESULT_DIR / "stage4_hard_classes.csv",
    index=False,
)
priority_confusions.to_csv(
    RESULT_DIR / "stage4_priority_confusions.csv",
    index=False,
)

safe_sync_output_tree(
    label="hard-class diagnostics"
)


## 16. 재학습 없이 먼저 확인할 것: ensemble weight sweep

현재 ensemble은 neural 70%, TF-IDF 30%라서 TF-IDF의 강점을 희석했습니다.  
기존 OOF probability만으로 TF-IDF 비중을 0.0부터 1.0까지 훑어볼 수 있습니다.

주의:

- 이 결과는 같은 OOF에서 weight를 선택하므로 약간 낙관적일 수 있습니다.
- 큰 폭의 개선이 아닌 미세한 차이는 Stage 5 채택 근거로 삼지 않습니다.
- 공식 test는 weight 선택에 사용하지 않습니다.


In [ ]:
def sweep_tfidf_weight(
    tfidf_probabilities,
    neural_probabilities,
    y_true,
    step=0.05,
):
    rows = []

    for tfidf_weight in np.arange(
        0.0,
        1.0 + step / 2,
        step,
    ):
        mixed = (
            tfidf_weight * tfidf_probabilities
            + (1.0 - tfidf_weight)
            * neural_probabilities
        )
        mixed = mixed / mixed.sum(
            axis=1,
            keepdims=True,
        )

        prediction = mixed.argmax(axis=1)
        metrics = compute_metrics(
            y_true,
            prediction,
        )

        rows.append(
            {
                "tfidf_weight": float(
                    tfidf_weight
                ),
                "neural_weight": float(
                    1.0 - tfidf_weight
                ),
                **metrics,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "macro_f1",
            ascending=False,
        )
        .reset_index(drop=True)
    )


if {
    "tfidf",
    "coarse",
}.issubset(probability_models):
    weight_sweep_df = sweep_tfidf_weight(
        probability_models["tfidf"],
        probability_models["coarse"],
        cv_labels,
    )

    display(
        weight_sweep_df.head(10).style.format(
            {
                "tfidf_weight": "{:.2f}",
                "neural_weight": "{:.2f}",
                "accuracy": "{:.4f}",
                "macro_precision": "{:.4f}",
                "macro_recall": "{:.4f}",
                "macro_f1": "{:.4f}",
            }
        )
    )

    weight_sweep_df.to_csv(
        RESULT_DIR
        / "stage4_ensemble_weight_sweep.csv",
        index=False,
    )

    safe_sync_output_tree(
        label="ensemble weight sweep"
    )


## 17. 특정 카테고리 처방: confusion-group specialist

### 핵심 아이디어

1. Global TF-IDF가 전체 20개 class probability를 만듭니다.
2. top-1과 top-2가 같은 confusion group 안에 있고 확률 차이가 작을 때만 specialist를 호출합니다.
3. Specialist는 그 group의 class만 구분하도록 별도로 학습합니다.
4. specialist 결과를 global probability와 섞어 최종 class를 결정합니다.

우선순위 group:

```text
religion:
alt.atheism
soc.religion.christian
talk.religion.misc

politics:
talk.politics.guns
talk.politics.mideast
talk.politics.misc

computer stack:
comp.graphics
comp.os.ms-windows.misc
comp.sys.ibm.pc.hardware
comp.sys.mac.hardware
comp.windows.x
sci.electronics

vehicles:
rec.autos
rec.motorcycles
```

Specialist도 반드시 outer fold의 training 문서만 이용해 학습해야 합니다.  
OOF validation 문서를 specialist 학습이나 threshold 선택에 섞으면 leakage가 됩니다.


In [ ]:
SPECIALIST_GROUPS = {
    "religion": [
        "alt.atheism",
        "soc.religion.christian",
        "talk.religion.misc",
    ],
    "politics": [
        "talk.politics.guns",
        "talk.politics.mideast",
        "talk.politics.misc",
    ],
    "computer_stack": [
        "comp.graphics",
        "comp.os.ms-windows.misc",
        "comp.sys.ibm.pc.hardware",
        "comp.sys.mac.hardware",
        "comp.windows.x",
        "sci.electronics",
    ],
    "vehicles": [
        "rec.autos",
        "rec.motorcycles",
    ],
}

ROUTER_MARGIN_THRESHOLD = 0.15


def analyze_specialist_routes(
    probabilities,
    y_true,
    class_names,
    groups,
    margin_threshold=0.15,
):
    top2 = np.argsort(
        probabilities,
        axis=1,
    )[:, -2:][:, ::-1]

    top1 = top2[:, 0]
    second = top2[:, 1]

    margin = (
        probabilities[
            np.arange(len(probabilities)),
            top1,
        ]
        - probabilities[
            np.arange(len(probabilities)),
            second,
        ]
    )

    global_error = top1 != y_true
    rows = []

    for group_name, group_classes in (
        groups.items()
    ):
        group_ids = {
            class_names.index(name)
            for name in group_classes
        }

        same_group_top2 = np.array(
            [
                (
                    int(first) in group_ids
                    and int(second_id)
                    in group_ids
                )
                for first, second_id in zip(
                    top1,
                    second,
                )
            ],
            dtype=bool,
        )

        routed = (
            same_group_top2
            & (margin <= margin_threshold)
        )

        routed_errors = (
            routed & global_error
        )

        rows.append(
            {
                "group": group_name,
                "route_candidates": int(
                    routed.sum()
                ),
                "errors_inside_candidates": int(
                    routed_errors.sum()
                ),
                "candidate_error_rate": (
                    float(
                        routed_errors.sum()
                        / max(routed.sum(), 1)
                    )
                ),
                "share_of_all_errors": (
                    float(
                        routed_errors.sum()
                        / max(
                            global_error.sum(),
                            1,
                        )
                    )
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "errors_inside_candidates",
            ascending=False,
        )
        .reset_index(drop=True)
    )


route_diagnostic_df = (
    analyze_specialist_routes(
        probability_models["tfidf"],
        cv_labels,
        target_names,
        SPECIALIST_GROUPS,
        margin_threshold=(
            ROUTER_MARGIN_THRESHOLD
        ),
    )
)

display(
    route_diagnostic_df.style.format(
        {
            "candidate_error_rate": "{:.2%}",
            "share_of_all_errors": "{:.2%}",
        }
    )
)

route_diagnostic_df.to_csv(
    RESULT_DIR
    / "stage5_specialist_route_diagnostic.csv",
    index=False,
)

safe_sync_output_tree(
    label="specialist route diagnostic"
)


## 18. Stage 5 권장 실험 순서

### A. Sparse global classifier 강화

현재 `SGDClassifier(loss="log_loss")` 대신 아래를 같은 fold에서 비교합니다.

1. `LinearSVC`
2. `LogisticRegression`
3. 필요할 때만 `CalibratedClassifierCV(LinearSVC)`

20 Newsgroups 같은 sparse text에서는 margin 기반 linear classifier가 SGD보다 안정적으로 좋은 경우가 많습니다.  
확률이 필요한 specialist/ensemble에는 calibration을 적용합니다.

### B. Confusion-group specialist reranker

가장 우선할 targeted 처방입니다.

```text
global TF-IDF
→ top-1/top-2와 margin 확인
→ 애매한 religion/politics/computer/vehicle 문서만 specialist 호출
→ global과 specialist score 결합
```

특히 다음 오류를 직접 겨냥합니다.

- `talk.religion.misc ↔ soc.religion.christian`
- `talk.religion.misc ↔ alt.atheism`
- `talk.politics.misc ↔ talk.politics.guns`
- `comp.sys.ibm.pc.hardware ↔ comp.os.ms-windows.misc`
- `rec.autos ↔ rec.motorcycles`

### C. Subject + body dual-channel

과제 조건상 raw header 사용이 허용될 때만 실험합니다.

- `Subject:`는 별도 TF-IDF branch로 사용
- `Newsgroups:`, `Xref:`, `Path:`처럼 label을 직접 노출할 수 있는 header는 반드시 제거
- Subject와 body score를 결합

이 방식은 broad `misc` class의 실제 질문 의도를 잡는 데 도움을 줄 수 있지만, header leakage 검사가 먼저입니다.

### D. Class-specific calibration

`talk.religion.misc`는 precision보다 recall이 특히 낮습니다.  
inner calibration split에서만 class bias 또는 temperature를 조정해 recall을 보정할 수 있습니다.

같은 OOF 정답을 보며 class bias를 직접 맞추면 과적합되므로 마지막 수단으로 둡니다.

### 당장 우선하지 않는 것

- 강한 text augmentation
- 더 큰 BiGRU
- coarse loss weight 반복 조정

이번 결과에서 sparse TF-IDF가 neural branch보다 명확히 우세했으므로, 먼저 sparse classifier와 specialist 구조에 계산량을 집중합니다.

### Stage 5 채택 기준

- TF-IDF 기준 OOF Macro F1 `0.7424`보다 최소 `+0.005`
- `talk.religion.misc` F1 또는 recall이 실질적으로 개선
- 다른 주요 class에서 큰 성능 붕괴가 없음
- 모든 선택은 OOF에서 완료
- 공식 test는 최종 설정에 한 번만 사용
